# Key Features
#### Dynamic NIFTY CSV parsing → avoids ParserError.
#### Dow Jones handled separately via Wikipedia API.
#### Multi-index tagging → symbols in multiple indices are combined with |.
#### Deduplication → keeps one row per (Stock Symbol, Stock Market).
#### Simple, maintainable structure — easy to extend for ISIN, sector, country later.

In [1]:
import pandas as pd
from pathlib import Path
from time import sleep
from io import StringIO
import requests


# =========================
# Directories
# =========================
RAW_DIR = Path("../Data/index_csv")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Index URLs
# =========================
URLS = {
    # US (DataHub)
    "S&P500": "https://datahub.io/core/s-and-p-500-companies/r/constituents.csv",
    # INDIA (NIFTY)
    "NIFTY50": "https://www.niftyindices.com/IndexConstituent/ind_nifty50list.csv",
    "NIFTY500": "https://www.niftyindices.com/IndexConstituent/ind_nifty500list.csv",
    "NIFTY_CHEMICALS": "https://niftyindices.com/IndexConstituent/ind_niftyChemicals_list.csv",
    "NIFTY_AUTO": "https://www.niftyindices.com/IndexConstituent/ind_niftyautolist.csv",
    "NIFTY_BANK": "https://www.niftyindices.com/IndexConstituent/ind_niftybanklist.csv",
    "NIFTY_FINANCIAL_SERVICES": "https://www.niftyindices.com/IndexConstituent/ind_niftyfinancelist.csv",
    "NIFTY_FMCG": "https://www.niftyindices.com/IndexConstituent/ind_niftyfmcglist.csv",
    "NIFTY_HEALTH_CARE": "https://www.niftyindices.com/IndexConstituent/ind_niftyhealthcarelist.csv",
    "NIFTY_IT": "https://www.niftyindices.com/IndexConstituent/ind_niftyitlist.csv",
}

# Browser-like headers (CRITICAL for NIFTY)
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/csv,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://www.niftyindices.com/",
    "Connection": "keep-alive",
}

# =========================
# Download with retries
# =========================
def download_with_retry(session, name, url, retries=3):
    for attempt in range(1, retries + 1):
        try:
            print(f"Downloading {name} , {url}, (attempt {attempt})")
            resp = session.get(url, headers=HEADERS, timeout=120)
            resp.raise_for_status()
            return resp.content
        except Exception as e:
            print(f"⚠️ Attempt {attempt} failed for {name}: {e}")
            sleep(5 * attempt)
    raise RuntimeError(f"❌ Failed after {retries} attempts: {name}")

# =========================
# Download all raw CSVs
# =========================
def download_raw():
    with requests.Session() as session:
        for name, url in URLS.items():
            content = download_with_retry(session, name, url)
            file_path = RAW_DIR / f"{name}.csv"
            file_path.write_bytes(content)
            print(f"✅ Saved raw file → {file_path}")

        # -------------------------
        # Dow Jones (Wikipedia)
        # -------------------------
        print("Downloading DOW_JONES from Wikipedia API")
        WIKI_API = "https://en.wikipedia.org/w/api.php"
        params = {"action": "parse", "page": "Dow_Jones_Industrial_Average",
                  "prop": "text", "format": "json"}
        r = session.get(WIKI_API, params=params, headers={"User-Agent": "IndexDataCollector/1.0"}, timeout=60)
        r.raise_for_status()
        html = r.json()["parse"]["text"]["*"]
        tables = pd.read_html(StringIO(html))
        djia_table = next(t for t in tables if {"Company", "Symbol"}.issubset(set(t.columns)))
        djia_df = djia_table[["Company", "Symbol"]]
        djia_df.columns = ["Stock Name", "Stock Symbol"]
        djia_path = RAW_DIR / "DOW_JONES.csv"
        djia_df.to_csv(djia_path, index=False)
        print(f"✅ Saved raw file → {djia_path}")

 

# =========================
# Download Dow Jones via Wikipedia
# =========================
def download_dow_jones():
    print("⬇️ Downloading DOW_JONES from Wikipedia API")
    WIKI_API = "https://en.wikipedia.org/w/api.php"
    headers = {"User-Agent": "IndexDataCollector/1.0"}
    params = {"action": "parse", "page": "Dow_Jones_Industrial_Average",
              "prop": "text", "format": "json"}
    r = requests.get(WIKI_API, params=params, headers=headers, timeout=60)
    r.raise_for_status()
    html = r.json()["parse"]["text"]["*"]
    tables = pd.read_html(StringIO(html))
    djia_table = next(t for t in tables if {"Company", "Symbol"}.issubset(set(t.columns)))
    #djia_df = djia_table[["Company", "Symbol"]]
    djia_df = djia_table[["Company", "Symbol"]].copy()
    djia_df.columns = ["Stock Name", "Stock Symbol"]
    djia_df.loc[:, "Index"] = "DOW JONES"
    path = RAW_DIR / "DOW_JONES.csv"
    djia_df.to_csv(path, index=False)
    print(f"✅ Saved DOW_2_JONES → {path}")
    return djia_df
    
OUT_DIR = Path("../Data/normalized_csv")
OUT_DIR.mkdir(exist_ok=True)

# =========================
# Process Index CSVs
# =========================
def process_sp500():
    df = pd.read_csv(RAW_DIR / "S&P500.csv")
    return pd.DataFrame({
        "Stock Name": df["Security"],
        "Stock Symbol": df["Symbol"],
        "Stock Market": "NYSE/NASDAQ",
        "INDEX": "S&P 500"
    })

def process_nasdaq():
    df = pd.read_csv(RAW_DIR / "NASDAQ.csv")
    return pd.DataFrame({
        "Stock Name": df["Company Name"],
        "Stock Symbol": df["Symbol"],
        "Stock Market": "NASDAQ",
        "INDEX": "NASDAQ"
    })

def process_nyse():
    csv_path = RAW_DIR / "NYSE.csv"
    if not csv_path.exists():
        print("⚠️ NYSE.csv not found, skipping NYSE index")
        return pd.DataFrame(columns=["Stock Name","Stock Symbol","Stock Market","INDEX"])
    df = pd.read_csv(csv_path)
    return pd.DataFrame({
        "Stock Name": df["Company Name"],
        "Stock Symbol": df["ACT Symbol"],
        "Stock Market": "NYSE",
        "INDEX": "NYSE"
    })

def process_nifty(index_name):
    csv_path = RAW_DIR / f"{index_name}.csv"
    if not csv_path.exists():
        print(f"⚠️ {index_name}.csv not found, skipping")
        return pd.DataFrame(columns=["Stock Name","Stock Symbol","Stock Market","INDEX"])
    df = pd.read_csv(csv_path)
    df = df[["Company Name", "Symbol"]]
    return pd.DataFrame({
        "Stock Name": df["Company Name"],
        "Stock Symbol": df["Symbol"],
        "Stock Market": "NSE",
        "INDEX": index_name.replace("NIFTY", "NIFTY ")
    })

def process_dow_jones():
    df = pd.read_csv(RAW_DIR / "DOW_JONES.csv")
    return pd.DataFrame({
        "Stock Name": df["Stock Name"],
        "Stock Symbol": df["Stock Symbol"],
        "Stock Market": "NYSE/NASDAQ",
        "INDEX": "DOW JONES"
    })




In [2]:
# =========================
# Main
# =========================
def main():
    download_raw()
    #download_dow_jones()

    # Process all indices
    datasets = [
        process_sp500(),
        #process_nasdaq(),
        #process_nyse(),
        process_nifty("NIFTY50"),
        process_nifty("NIFTY500"),
        process_dow_jones(),
    ]

    # Combine, deduplicate, and aggregate
    final_df = pd.concat(datasets, ignore_index=True)
    final_df.dropna(inplace=True)
    final_df = (final_df.groupby(["Stock Symbol", "Stock Market"], as_index=False)
                .agg({"Stock Name": "first",
                      "INDEX": lambda x: "|".join(sorted(set(x)))}))

    out_file = OUT_DIR / "ALL_INDEX_STOCKS.csv"
    final_df.to_csv(out_file, index=False)

    print(f"✅ Normalized file created → {out_file}")
    print(f"Total rows: {len(final_df)}")

if __name__ == "__main__":
    main()

✅ Saved raw file → ..\Data\index_csv\S&P500.csv
✅ Saved raw file → ..\Data\index_csv\NIFTY50.csv
✅ Saved raw file → ..\Data\index_csv\NIFTY500.csv
✅ Saved raw file → ..\Data\index_csv\NIFTY_CHEMICALS.csv
✅ Saved raw file → ..\Data\index_csv\NIFTY_AUTO.csv
✅ Saved raw file → ..\Data\index_csv\NIFTY_BANK.csv
✅ Saved raw file → ..\Data\index_csv\NIFTY_FINANCIAL_SERVICES.csv
✅ Saved raw file → ..\Data\index_csv\NIFTY_FMCG.csv
✅ Saved raw file → ..\Data\index_csv\NIFTY_HEALTH_CARE.csv
✅ Saved raw file → ..\Data\index_csv\NIFTY_IT.csv
✅ Saved raw file → ..\Data\index_csv\DOW_JONES.csv
✅ Normalized file created → ..\Data\normalized_csv\ALL_INDEX_STOCKS.csv
Total rows: 1004


In [3]:
# download_dow_jones()

In [4]:
#download_raw()

# INDIA STOCK MARKET LISTED COMPINIES

In [7]:
import requests
import pandas as pd
from pathlib import Path

# ================= CONFIG ================= #

DOWNLOAD_DIR = Path("../Data/index_csv")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "text/csv,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# ================= NSE ================= #

def download_nse():
    print("Downloading NSE listed companies...")

    url = "https://archives.nseindia.com/content/equities/EQUITY_L.csv"

    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()

    file_path = DOWNLOAD_DIR / "NSE_listed_companies.csv"

    file_path.write_bytes(r.content)

    print(f"NSE saved to: {file_path.resolve()}")
    return pd.read_csv(file_path)


# ================= BSE ================= #

def download_bse():
    print("Downloading BSE listed companies...")

    url = "https://www.bseindia.com/download/BhavCopy/Equity/EQ_ISINCODE.csv"

    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()

    file_path = DOWNLOAD_DIR / "BSE_listed_companies.csv"
    file_path.write_bytes(r.content)

    print(f"BSE saved to: {file_path.resolve()}")
    return pd.read_csv(file_path)


# ================= MAIN ================= #

if __name__ == "__main__":

    try:
        nse_df = download_nse()
        print(nse_df.head())
    except Exception as e:
        print("NSE download failed:", e)

    try:
        bse_df = download_bse()
        print(bse_df.head())
    except Exception as e:
        print("BSE download failed:", e)

    print("\nAll downloads complete.")

NSE saved to: C:\ljmu\ljmu\Data\index_csv\NSE_listed_companies.csv
       SYMBOL                           NAME OF COMPANY  SERIES  \
0   20MICRONS                        20 Microns Limited      EQ   
1  21STCENMGM  21st Century Management Services Limited      EQ   
2      360ONE                       360 ONE WAM LIMITED      EQ   
3   3IINFOLTD                       3i Infotech Limited      EQ   
4     3MINDIA                          3M India Limited      EQ   

   DATE OF LISTING   PAID UP VALUE   MARKET LOT   ISIN NUMBER   FACE VALUE  
0      06-OCT-2008               5            1  INE144J01027            5  
1      03-MAY-1995              10            1  INE253B01015           10  
2      19-SEP-2019               1            1  INE466L01038            1  
3      22-OCT-2021              10            1  INE748C01038           10  
4      13-AUG-2004              10            1  INE470A01017           10  
BSE download failed: 404 Client Error: Not Found for url: https://ww